# UNDERTONE - cascaded ASR+LLM control

**Not one of the thirteen.** This is the validity check that preempts the
obvious objection: *isn't this just ASR plus a language model?*

It runs exactly that system through exactly the same protocol - same items, same
option shuffling, same ladder windows, same letter-logit scoring - so any gap
between it and the audio models is the modality, not the harness. That is why it
is a `ModelAdapter` and not a separate script.

## What its failure means, per category

| | why the cascade cannot answer |
|---|---|
| P1 / P2 | Whisper never wrote the muttered or overlapped words down. No language model behind it could have found them. |
| P4 | ASR normalises a self-repair away - "twenty twelve milligrams" loses the seam that says which value survived. |
| C1 | Hesitancy is not in the words at all. |

`01_build_item_pack` measures this directly with `needle_recovery`: the share of
items whose answer never appears in the ASR transcript. That is stronger evidence
than this model's score, because it does not depend on how good the language
model is.

## One detail that matters

VAD filtering is **off**. Voice-activity detection would drop exactly the quiet
spans P1 is about, which would flatter the cascade by never asking it the hard
question.

Each language is swept separately so Whisper is given the right language code
rather than guessing - the most favourable setting for the control.


In [ ]:
# Pinned for this model. If `load()` fails, this cell is the first thing to change.
%pip install -q "transformers==4.57.1"
%pip install -q "accelerate>=1.0.0"
%pip install -q "librosa>=0.10.2"
%pip install -q "soundfile>=0.12.1"
%pip install -q "faster-whisper>=1.0.0"
print("--- resolved versions (freeze these before the paper run) ---")
import importlib.metadata as md
for pkg in ["transformers", "accelerate", "torch", "librosa"]:
    try:
        print(f"{pkg:14s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:14s} not installed")

In [ ]:
import os, random, sys, json
import numpy as np, torch

SEED = 20260904
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Weights go to /kaggle/temp: scratch, and it does NOT count against the 20 GB
# /kaggle/working output cap. A 16-18 GB checkpoint in /kaggle/working would
# fail the commit at the end of the session.
os.environ.setdefault("HF_HOME", "/kaggle/temp/hf")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
# Long audio prompts allocate in large irregular blocks; without this the T4
# fragments and OOMs with a gigabyte nominally free.
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"cuda:{i}  {p.name}  {p.total_memory/1e9:.1f} GB  sm{p.major}{p.minor}")
if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] < 8:
    print("\nsm < 80: no bf16 compute and no flash-attention-2. "
          "Every adapter loads in fp16 for this reason.")

In [ ]:
REPO_URL = "https://github.com/DeepanIsCool/longaudiobench.git"
REPO_REF = "undertone"   # pin to a commit sha before the paper run

import subprocess, shutil, os, sys
if os.path.exists("/kaggle/working/longaudiobench"):
    shutil.rmtree("/kaggle/working/longaudiobench")
for attempt in range(3):
    rc = subprocess.call(["git", "clone", "--depth", "1", "--branch", REPO_REF,
                          REPO_URL, "/kaggle/working/longaudiobench"])
    if rc == 0:
        break
else:
    raise RuntimeError("could not clone the benchmark repo")

sys.path.insert(0, "/kaggle/working/longaudiobench")
import importlib; importlib.invalidate_caches()

from undertone import ItemPack, adapters, env, runner, scoring
print("adapters registered:", len(adapters.list_adapters()))

if env.export_hf_token():
    print("HF token resolved")
elif globals().get("GATED"):
    raise RuntimeError(
        "this model is gated and no token was found. Add a Kaggle secret named "
        "HF_TOKEN, or write the token to .hf_token at the repo root.")

hw = env.resolve_hardware()
print(f"hardware: {hw.detail}  dtype={hw.dtype}  signature={hw.signature}")
print(f"versions: {env.versions()}")
# Every result row is stamped with this signature. The analysis refuses to put
# two signatures in one table -- a benchmark whose rows came from different
# backends compares machines, not models.

In [ ]:
# The item pack is built once on CPU (notebook 01) and attached as a Kaggle
# Dataset, so a model sweep never re-harvests audio.
# Locate the pack rather than assume the mount name. Kaggle derives the input
# directory from the dataset slug, and hardcoding "/kaggle/input/undertone-item-pack" failed
# twice against a dataset that was correctly attached the whole time.
import glob

candidates = sorted(glob.glob("/kaggle/input/*/item_pack.jsonl")
                    + glob.glob("/kaggle/working/item_pack/item_pack.jsonl"))
if not candidates:
    listing = sorted(glob.glob("/kaggle/input/*")) or ["(nothing mounted)"]
    raise FileNotFoundError(
        "no item_pack.jsonl under /kaggle/input. Attach the 'undertone-item-pack' dataset "
        f"(Add Input -> Datasets), or run 01_build_item_pack first. "
        f"Currently mounted: {listing}")
PACK_DIR = os.path.dirname(candidates[0])
print(f"item pack: {PACK_DIR}")

pack = ItemPack.load(os.path.join(PACK_DIR, "item_pack.jsonl"))
print(f"{len(pack)} items from {len({i.recording_id for i in pack})} recordings")
for key, n in sorted(pack.counts("lang", "category").items()):
    print(f"  {key[0]}  {key[1]}  n={n}")

# What this model can and cannot ingest, stated before the run rather than
# discovered from a table of zeros afterwards.
from undertone.ladder import CONDITIONS, window_for
print(f"\ndocumented ceiling: {7200.0:.0f} s")
for cond in CONDITIONS:
    over = sum(1 for i in pack if window_for(i, cond).seconds > 7200.0)
    print(f"  {cond}: {over}/{len(pack)} cells exceed it -> truncated, not scored as 0")

In [ ]:
from undertone.adapters.cascaded import CascadedWhisperLLM
from undertone.ladder import CONDITIONS

OUT = "/kaggle/working/results/cascaded_whisper_llm.jsonl"

for lang in ("en", "hi", "bn"):
    subset = pack.filter(lang=lang)
    if not len(subset):
        continue
    print(f"\n=== {lang}: {len(subset)} items ===")
    control = CascadedWhisperLLM(lang=lang)
    runner.run_model(control, subset, out_path=OUT, conditions=CONDITIONS,
                     seed=SEED, run_id="pilot", audio_root=PACK_DIR)
    control.unload()

rows = runner.load_rows(OUT)
usable = runner.scorable(rows)
print(f"\n{len(usable)} scorable rows")
for cat in ["P1", "P2", "P3", "P4", "C1"]:
    cell = [r for r in usable if r["category"] == cat and r["condition"] == "L3"]
    if cell:
        s = scoring.summarize(cell)
        print(f"  {cat}  n={s['n']:3d}  acc={s['accuracy']:.3f}  "
              f"salience={s['salience_trap_rate']:.3f}")